# scope
**Prerequisites:** function_basics, parameters_and_arguments, return_values

**Outcome:** After this notebook you will know exactly where Python looks for a name when you use it, why the same name can mean different things in different places, and how to avoid every class of scope-related bug.

## The Problem

In [ ]:
status = "idle"

def start_pipeline():
    status = "running"     # is this changing the global or creating a new local?
    print(f"inside : {status}")

start_pipeline()
print(f"outside: {status}")   # what is printed here and why?

## The Concept

Scope defines where a name is visible and where Python looks for it when you use it. Python follows a fixed lookup order called LEGB: Local first, then Enclosing, then Global, then Built-in. Local is the current function. Enclosing is any outer function containing this one. Global is the module level. Built-in is Python's own namespace of functions like `len` and `print`. Python searches these layers in order and stops at the first match. Assignment inside a function always creates a local name unless you explicitly declare otherwise with `global` or `nonlocal`.

## Minimal Example

In [ ]:
region = "us-east"          # Global

def show_region():
    print(region)            # Local has no 'region' — found in Global

show_region()                # us-east

def override_region():
    region = "eu-west"       # new Local name — does not touch Global
    print(region)            # eu-west

override_region()
print(region)                # us-east — Global unchanged

## How Python Does It

Python resolves names at runtime using the LEGB rule. Each function call creates a new local namespace — a dictionary of name-to-object bindings. When the function finishes that dictionary is destroyed. At compile time Python decides whether a name inside a function is local by checking whether the function contains any assignment to that name. If it does, the name is treated as local throughout the entire function body — even before the assignment line. This is why reading a name before assigning it inside the same function raises `UnboundLocalError`, not `NameError`.

In [ ]:
import dis

x = 10

def read_global():
    return x          # LOAD_GLOBAL

def read_local():
    x = 20
    return x          # LOAD_FAST

print("--- read_global ---")
dis.dis(read_global)
print("--- read_local ---")
dis.dis(read_local)

## Building Up

In [ ]:
# L — Local scope
def compute(value):
    result = value * 2   # result is Local — only visible inside compute
    return result

print(compute(5))
try:
    print(result)         # NameError — result does not exist outside
except NameError as e:
    print(e)

In [ ]:
# E — Enclosing scope
def outer():
    tag = "pipeline"         # Enclosing for inner()

    def inner():
        print(tag)           # found in Enclosing — not Local, not Global

    inner()

outer()

In [ ]:
# G — Global scope
MAX_RETRIES = 3              # Global

def should_retry(attempt):
    return attempt < MAX_RETRIES   # found in Global

print(should_retry(1))   # True
print(should_retry(3))   # False

In [ ]:
# B — Built-in scope
def count_jobs(jobs):
    return len(jobs)    # len found in Built-in — you never defined it

print(count_jobs(["job_1", "job_2", "job_3"]))

In [ ]:
# global keyword — explicitly modify a global from inside a function
active_region = "us-east"

def switch_region(new_region):
    global active_region     # declare intent to modify the global
    active_region = new_region

print(active_region)         # us-east
switch_region("eu-west")
print(active_region)         # eu-west

In [ ]:
# nonlocal keyword — modify an enclosing (not global) variable
def make_counter():
    count = 0

    def increment():
        nonlocal count       # modify enclosing count, not create a new local
        count += 1
        return count

    return increment

counter = make_counter()
print(counter())   # 1
print(counter())   # 2
print(counter())   # 3

In [ ]:
# full LEGB in one example
tag = "global_tag"                    # G

def outer():
    tag = "enclosing_tag"             # E

    def inner():
        tag = "local_tag"             # L
        print(tag)                    # L wins
        print(len)                    # B — built-in

    inner()
    print(tag)                        # E — inner's local does not affect outer

outer()
print(tag)                            # G — unchanged

## Where It Breaks

In [ ]:
# UnboundLocalError — Python decides 'count' is local because of the assignment
# but you try to read it before assigning
count = 10

def increment():
    count += 1        # assignment makes count local — but it has no local value yet
    return count

try:
    increment()
except UnboundLocalError as e:
    print(e)

In [ ]:
# shadowing a built-in — silently breaks everything that relied on it
list = [1, 2, 3]      # 'list' now shadows the built-in type

try:
    result = list(range(5))   # TypeError — list is now [1, 2, 3], not the type
except TypeError as e:
    print(e)

del list              # restore the built-in
print(list(range(5))) # works again

In [ ]:
# loop variable leaks into enclosing scope in Python (unlike some languages)
for i in range(5):
    pass

print(i)   # 4 — i still exists after the loop
# this is not a bug in Python but it surprises developers from other languages

## The Fix

In [ ]:
# fix UnboundLocalError — use global or pass the value as an argument
count = 10

# option 1: global
def increment_global():
    global count
    count += 1

increment_global()
print(count)   # 11

# option 2: pass and return — preferred, no hidden state
def increment(n):
    return n + 1

count = increment(count)
print(count)   # 12

In [ ]:
# fix shadowing — never use built-in names for variables
# bad names: list, dict, set, type, id, input, filter, map, min, max, sum

# bad
# list = ["job_1", "job_2"]

# good
job_list = ["job_1", "job_2"]
print(job_list)

## In a Real System

In [ ]:
# module-level config read by all functions — correct use of global scope
# functions read it but never assign to it — no global keyword needed
DEFAULT_REGION  = "us-east"
MAX_RETRIES     = 3
BATCH_SIZE      = 100

def build_job(job_id, region=None):
    return {
        "id":      job_id,
        "region":  region or DEFAULT_REGION,   # reads Global — fine
        "retries": MAX_RETRIES                 # reads Global — fine
    }

def process_batch(records):
    for i in range(0, len(records), BATCH_SIZE):   # reads Global — fine
        batch = records[i : i + BATCH_SIZE]
        print(f"processing batch of {len(batch)}")

print(build_job("job_1"))
process_batch(list(range(250)))

## Performance

Local variable lookup (`LOAD_FAST`) is faster than global lookup (`LOAD_GLOBAL`) because locals are stored in a fixed-size array indexed by integer, while globals require a dictionary key lookup. In tight inner loops that reference a global many times, copying it to a local variable at the start of the function is a real micro-optimisation. Built-in lookup is slowest — it searches three dictionaries before finding the name. Shadowing a built-in with a local name actually speeds up that one lookup while breaking everything else.

## Anti-Pattern

In [ ]:
# anti-pattern: using global variables as function state
error_count = 0

def process(record):
    global error_count
    if not record.get("id"):
        error_count += 1     # hidden side effect — impossible to test in isolation

process({"id": ""})
process({"id": ""})
print(error_count)   # 2 — but only if nothing else touched error_count

# correct: pass state in, return state out — no hidden globals
def process(record, error_count):
    if not record.get("id"):
        return error_count + 1
    return error_count

error_count = 0
error_count = process({"id": ""}, error_count)
error_count = process({"id": ""}, error_count)
print(error_count)   # 2 — explicit, testable, no shared state

## Interview Signals

- What does LEGB stand for and in what order does Python search each layer?
- What is the difference between `global` and `nonlocal`?
- Why does assigning to a name inside a function make the entire function treat that name as local?
- Why is local variable lookup faster than global variable lookup in CPython?

## Exercise

In [ ]:
def make_pipeline_tracker():
    """
    Returns two functions: record_job and get_summary.

    record_job(job_id, status) — records a job result.
    get_summary()              — returns a dict with keys:
                                 'total', 'done', 'failed'

    The two functions must share state using enclosing scope
    (nonlocal or a shared mutable container).
    No global variables allowed.

    Bug: the function bodies are missing — implement them.
    """
    pass


record_job, get_summary = make_pipeline_tracker()

record_job("job_1", "done")
record_job("job_2", "failed")
record_job("job_3", "done")
record_job("job_4", "failed")
record_job("job_5", "done")

summary = get_summary()

assert summary["total"]  == 5, f"got {summary['total']}"
assert summary["done"]   == 3, f"got {summary['done']}"
assert summary["failed"] == 2, f"got {summary['failed']}"

# two trackers must not share state
record_job_b, get_summary_b = make_pipeline_tracker()
record_job_b("job_x", "done")
assert get_summary_b()["total"] == 1
assert get_summary()["total"]   == 5   # original tracker unchanged

print("all assertions passed")